# LTI Systems and Stability Margin

In undergraduate control systems classes you learn how to analyze an LTI system and evaluate its stability using the Nyquist plot, which plots the real and imaginary parts of the system's frequency response.

Students are generally introduced to two measures of stability: gain margin and phase margin.

## What is gain margin?

Gain margin is how much the loop gain can be multiplied before the system goes unstable. On the Nyquist plot, find where the curve crosses the negative real axis (where the phase is −180°). If it crosses at $-a$, multiplying the gain by $1/a$ stretches the curve until that crossing lands exactly on the critical point $-1$. So:

$$GM = \frac{1}{a} \qquad GM_{dB} = 20\log_{10}\frac{1}{a}$$

For example, a crossing at $-0.5$ gives $GM = 2$, or about 6 dB. The gain margin only looks **along the real axis**. It says nothing about how close the curve gets to $-1$ anywhere else. If the curve never crosses the negative real axis, as in the example below, the gain margin is infinite.

Gain margin can be interpreted as a control system's tolerance to extra amplification. Some sources of gain error:

- **Poor calibration.** A command of 1.0 might produce a response of 1.2 at the actuator. Errors like this are unavoidable, but with enough gain margin they can be tolerated.
- **The plant changes with operating conditions.** An aircraft control surface produces force in proportion to dynamic pressure, so the same deflection is far more powerful at high speed and low altitude than at low speed and high altitude. One controller has to handle the whole flight envelope, so the loop gain it sees can change by a factor of several.
- **Load or inertia changes.** A robot arm picking up a payload, or a vehicle with a full tank versus an empty one, presents a different effective gain to the same controller.
- **Part-to-part variation and aging.** Motor torque constants, amplifier gains and sensor scale factors typically vary by ±5–10% between units, and they drift with temperature and wear. A design tuned on one prototype has to work on every unit you build.
- **Supply voltage.** A motor driven by PWM from a battery loses actuator gain as the battery drains, and gains it back when you plug in a fresh one.

## What is phase margin?

Phase margin is how much extra phase lag the loop can take before the system goes unstable. On the Nyquist plot, find where the curve crosses the unit circle, the frequency $\omega_c$ where the loop gain is exactly 1. Adding lag rotates the curve clockwise around the origin without changing its size. The phase margin is the angle the curve can rotate before that crossing lands on the critical point $-1$:

$$PM = 180° + \angle L(j\omega_c)$$

For example, if the curve crosses the unit circle at −120°, the phase margin is 60°. Like gain margin, phase margin only looks at **one point**: where the curve crosses the unit circle. It says nothing about how close the curve gets to $-1$ anywhere else.

Phase margin can be interpreted as a control system's tolerance to lag. Lag can be introduced in many ways. Some examples:

- **Computation and actuation delay.** It takes time for a computer to calculate the control output, and even more time for the actuator to respond. This delay can be accounted for with phase margin rather than baked into the plant model itself.
- **Sampling.** A digital controller only sees the measurement once per sample period $T$, and the zero-order hold keeps its output constant in between. Together they act like a delay of about $T/2$. A pure delay $\tau$ costs $\omega_c \tau$ radians of phase at crossover, so a faster loop loses more margin to the same delay.
- **Sensor filtering.** Noisy sensors are usually low-pass filtered, and anti-aliasing filters sit in front of every ADC. Each filter adds phase lag, and that lag is often significant near crossover even when the filter's cutoff looks comfortably high.
- **Communication delays.** Networked and remote systems (CAN bus, Ethernet, teleoperation) add transport delay between the sensor, the controller and the actuator. Process plants have physical transport delays too, such as fluid travelling down a pipe or material moving along a conveyor.
- **Unmodeled dynamics.** Real actuators have limited bandwidth: a servo motor or hydraulic valve doesn't respond instantly. Structures also flex. These dynamics are often left out of the plant model, but each one adds lag near crossover.

## Is this good enough in the real world? No!

These concepts are a great introduction to LTI stability, but are they enough to analyze a system in the real world? Unfortunately, the answer is a resounding no.

Both tell you how well a system tolerates gain **or** phase error in isolation, but not in combination. In real systems you will almost always be dealing with errors that affect both gain and phase.

## Vector margin

The fix is to stop measuring at one special point and measure the distance to −1 everywhere. The **vector margin** (VM) is the distance from the critical point −1 to the closest point on the Nyquist curve:

$$VM = \min_\omega \left|\,1 + L(j\omega)\,\right|$$

Picture a circle of radius VM centred on −1. The Nyquist curve never enters it. For the system to go unstable, some error has to drag a point on the curve all the way onto −1, and the circle tells you how far away from −1 every point starts.

Because the curve has to stay outside that circle, a vector margin guarantees a minimum gain and phase margin:

$$GM \ge \frac{1}{1 - VM} \qquad PM \ge 2\arcsin\frac{VM}{2}$$

More importantly, it covers gain and phase errors **at the same time**. Write a combined gain error $k$ and phase lag $\varphi$ as one complex factor $f = k e^{-j\varphi}$ multiplying the loop. The system stays stable for every $f$ with

$$\left|\frac{1}{f} - 1\right| < VM$$

This is a whole region of simultaneous gain and phase errors, and the two bounds above are its edges along each axis. Gain and phase margin can't give you anything like it: each one only describes errors in one direction, with the other error at zero.

The reverse isn't true: big gain and phase margins don't guarantee a big vector margin. A common design target is VM ≥ 0.5, which guarantees at least 6 dB of gain margin and 29° of phase margin.

Vector margin is also $1/M_s$, where $M_s$ is the peak of the sensitivity function $S = 1/(1+L)$. You'll often see robustness requirements written as a limit on $M_s$ instead.


In [7]:
import numpy as np
import control as ct
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

In [8]:
s = ct.tf("s")
 
# ---------------- USER INPUT: plant P and controller C ----------------
P = 9 / ((s + 2) * (s**2 + 0.48 * s + 9))
C = (4 / 9) * (s**2 + 1.2 * s + 9) / s
# ----------------------------------------------------------------------
 
L = P * C
w = np.logspace(-3, 3, 20000)
L_jw = ct.frequency_response(L, w).complex

# An example

This LTI system has an infinite gain margin and a high phase margin. But as you can see, the frequency response curve actually gets much closer to the critical point than either of these numbers would indicate. Its vector margin is only 0.336. On its own, that would guarantee just 3.6 dB of gain margin and 19° of phase margin, a long way from the infinite gain margin and 61° phase margin the classic measures report.

In [3]:
def margins(sys):
    """GM, PM, VM and the Nyquist points where they occur (via ct.stability_margins)."""
    gm, pm, vm, wpc, wgc, wvm = ct.stability_margins(sys)
    ev = lambda wx: complex(sys(1j * wx)) if np.isfinite(wx) else None
    return dict(gm=gm, pm=pm, vm=vm, gm_pt=ev(wpc), pm_pt=ev(wgc), vm_pt=ev(wvm))


def label(m, name, z=0):
    """Margins text; z = closed-loop RHP poles. VM is reported as 0 when unstable."""
    gm = "inf" if not np.isfinite(m["gm"]) else f"{20*np.log10(m['gm']):.1f} dB"
    pm = "inf" if not np.isfinite(m["pm"]) else f"{m['pm']:.1f}°"
    vm = m["vm"] if z == 0 else 0.0
    return f"{name}: GM={gm}, PM={pm}, VM={vm:.3f}"


def margin_traces(m, color):
    """Plotly traces marking the GM, PM and VM points."""
    t = []
    if m["gm_pt"] is not None:
        z = m["gm_pt"]
        t.append(go.Scatter(x=[z.real], y=[z.imag], mode="markers", name="GM point",
                            marker=dict(symbol="square", size=10, color=color)))
    if m["pm_pt"] is not None:
        z = m["pm_pt"]
        t.append(go.Scatter(x=[0, z.real], y=[0, z.imag], mode="lines", showlegend=False,
                            line=dict(color=color, width=1, dash="dash"), hoverinfo="skip"))
        t.append(go.Scatter(x=[z.real], y=[z.imag], mode="markers", name="PM point",
                            marker=dict(symbol="triangle-up", size=11, color=color)))
    z, th = m["vm_pt"], np.linspace(0, 2 * np.pi, 200)
    t.append(go.Scatter(x=-1 + m["vm"] * np.cos(th), y=m["vm"] * np.sin(th), mode="lines",
                        name="VM circle", line=dict(color=color, width=1, dash="dot"),
                        hoverinfo="skip"))
    t.append(go.Scatter(x=[-1, z.real], y=[0, z.imag], mode="lines", showlegend=False,
                        line=dict(color=color, width=1.5), hoverinfo="skip"))
    t.append(go.Scatter(x=[z.real], y=[z.imag], mode="markers", name="Closest to −1 (VM)",
                        marker=dict(size=9, color=color)))
    return t


def nyquist_fig(Ljw, m, color, title):
    """Nyquist plot of Ljw with unit circle, −1 point and margin markers."""
    th = np.linspace(0, 2 * np.pi, 400)
    fig = go.Figure([
        go.Scatter(x=np.cos(th), y=np.sin(th), mode="lines", name="Unit circle",
                   line=dict(color="lightgray", width=1), hoverinfo="skip"),
        go.Scatter(x=Ljw.real, y=-Ljw.imag, mode="lines", name="ω < 0",
                   line=dict(color=color, width=1), opacity=0.3, hoverinfo="skip"),
        go.Scatter(x=Ljw.real, y=Ljw.imag, mode="lines", name="L(jω)",
                   line=dict(color=color, width=2), customdata=w,
                   hovertemplate="ω=%{customdata:.3g}<br>%{x:.3f} %{y:+.3f}j<extra></extra>"),
        go.Scatter(x=[-1], y=[0], mode="markers", name="−1",
                   marker=dict(symbol="cross-thin", size=14, line=dict(width=2, color="black"))),
        *margin_traces(m, color),
    ])
    fig.update_layout(title=dict(text=title, font=dict(size=14)), width=650, height=650,
                      template="plotly_white", legend=dict(x=1.02, y=1))
    fig.update_xaxes(title="Re", range=[-3, 1.5], zeroline=True)
    fig.update_yaxes(title="Im", range=[-3, 1.5], zeroline=True, scaleanchor="x", scaleratio=1)
    return fig


# Nyquist: Z = P - N, with N the CCW encirclements of -1 by L over the full contour.
# By conjugate symmetry (plus the arc closing the contour at w -> 0), the total winding
# of 1 + L is 2 * (final unwrapped angle) when the angle starts on the right branch.
# Calibrate that starting branch from the nominal loop, whose Z we know exactly.
P_ol = int(np.sum(ct.poles(L).real > 1e-9))
Z_nom = int(np.sum(ct.poles(ct.feedback(L, 1)).real > 0))
_a = np.unwrap(np.angle(1 + L_jw))
a0_nom = np.pi * (P_ol - Z_nom) - (_a[-1] - _a[0])


def closed_loop_rhp_poles(Ljw, phase_err_deg=0.0):
    """Closed-loop RHP pole count for the loop k*exp(-j*phase_err)*L via the Nyquist criterion."""
    a = np.unwrap(np.angle(1 + Ljw))
    a += 2 * np.pi * np.round((a0_nom - np.radians(phase_err_deg) - a[0]) / (2 * np.pi))
    return P_ol - int(round(a[-1] / np.pi))


def phase_err_tf(k, phase_err_deg):
    """Rational stand-in for exp(-j*phase_err) that has that phase at the crossover of k*L.

    Lag -> time delay (Pade); lead -> lead compensator with its peak lead and unity gain there.
    """
    wc = ct.stability_margins(k * L)[4]
    phi = np.radians(phase_err_deg)
    if not np.isfinite(wc) or abs(phi) < 1e-6:
        return ct.tf(1, 1)
    if phi > 0:
        return ct.tf(*ct.pade(phi / wc, 6))
    sn = np.sin(min(-phi, np.radians(85)))  # a single lead stage tops out below 90°
    alpha = (1 - sn) / (1 + sn)
    T = 1 / (wc * np.sqrt(alpha))
    return np.sqrt(alpha) * ct.tf([T, 1], [alpha * T, 1])


t_step = np.linspace(0, 40, 2000)
T_nom = ct.feedback(L, 1)
y_nom = ct.step_response(T_nom, t_step).outputs


def step_fig(Tp, title):
    """Closed-loop step response of Tp, with the original response for reference."""
    fig = go.Figure([
        go.Scatter(x=t_step, y=y_nom, mode="lines", name="Original",
                   line=dict(color="#1f77b4", width=1.5), opacity=0.5),
        go.Scatter(x=t_step, y=ct.step_response(Tp, t_step).outputs, mode="lines",
                   name="Perturbed", line=dict(color="#d62728", width=2)),
    ])
    fig.add_hline(y=1, line=dict(color="lightgray", dash="dash"))
    fig.update_layout(title=dict(text=title, font=dict(size=14)), width=650, height=400,
                      template="plotly_white", legend=dict(x=1.02, y=1))
    fig.update_xaxes(title="t (s)")
    fig.update_yaxes(title="y", range=[-0.5, 2.5])
    return fig

In [4]:
# ---------------- original plot ----------------
print("Nominal closed-loop poles:", np.round(ct.poles(ct.feedback(L, 1)), 3))

m0 = margins(L)
nyquist_fig(L_jw, m0, "#1f77b4", "Original — " + label(m0, "L", Z_nom)).show()

Nominal closed-loop poles: [-0.115+3.121j -0.115-3.121j -1.125+1.557j -1.125-1.557j]


Below is a plot that lets you add gain and phase error and see how they change the remaining gain, phase and vector margins. A step response underneath shows the effect in the time domain.

I recommend starting by adding phase error (try 6 degrees). You'll see that the gain margin is no longer infinite, far from it.

Then start adding gain error (try 6.5 dB). The system approaches instability and eventually goes unstable, despite its strong initial gain and phase margins.

Now compare the vector margins. The original loop's VM was only 0.336 to begin with, and as you add error the perturbed VM shrinks towards zero. Gain and phase margin said this loop was very safe. Vector margin was warning you all along.


In [5]:
# ---------------- perturbed plot ----------------
s_k = widgets.FloatSlider(value=0, min=-20, max=20, step=0.5, description="Gain err (dB)",
                          continuous_update=False, style={"description_width": "initial"},
                          layout=widgets.Layout(width="500px"))
s_p = widgets.FloatSlider(value=0, min=-90, max=90, step=1, description="Phase err (°)",
                          continuous_update=False, style={"description_width": "initial"},
                          layout=widgets.Layout(width="500px"))
b_reset = widgets.Button(description="Reset")
out = widgets.Output()
out_step = widgets.Output()

In [6]:
def update(_=None):
    k = 10 ** (s_k.value / 20)
    Lp_jw = k * np.exp(-1j * np.radians(s_p.value)) * L_jw
    Lp = ct.frd(Lp_jw, w, smooth=True)  # perturbed loop as frequency-response data

    mp = margins(Lp)
    z = closed_loop_rhp_poles(Lp_jw, s_p.value)
    title = "Perturbed — " + label(mp, "L", z) + (f"  → <b>UNSTABLE</b> ({z} RHP poles)" if z else "")
    with out:
        out.clear_output(wait=True)
        nyquist_fig(Lp_jw, mp, "#d62728", title).show()

    # step response of the rational stand-in for the perturbation
    Tp = ct.feedback(k * phase_err_tf(k, s_p.value) * L, 1)
    stable = np.all(ct.poles(Tp).real < 0)
    with out_step:
        out_step.clear_output(wait=True)
        step_fig(Tp, "Step response — " + ("stable" if stable else "<b>UNSTABLE</b>")).show()


def reset(_):
    s_k.value, s_p.value = 0, 0


s_k.observe(update, names="value")
s_p.observe(update, names="value")
b_reset.on_click(reset)

display(widgets.VBox([widgets.HBox([widgets.VBox([s_k, s_p]), b_reset]), out, out_step]))
update()

In conclusion, the gain and phase margins students learn about are extremely instructive and important. However, they measure robustness at only one point each. To understand the stability of an LTI system under real conditions, where gain and phase errors arrive together, you need a measure like the vector margin.

Vector margin isn't the whole story either. Two caveats:

- **It's a guarantee, not a boundary.** Errors inside the VM region are always safe, but plenty of errors outside it are safe too. Try a large phase lead or a big gain reduction in the plot above: the loop stays stable even though both are well beyond what VM promises. A small VM tells you the system is fragile and in which direction, not exactly where the edge is.
- **It needs the error to stay small at every frequency.** The guarantee holds as long as the error stays inside the VM region at each frequency. A time delay doesn't: its phase lag $\omega\tau$ keeps growing with frequency, so it always leaves the region eventually. For delays, engineers check the delay margin directly. Disk margins generalize the vector margin idea and are what many robust-control tools report.
